# 02 — Session check and product orientation

**Goal:** discover from the file (not from its name) which frequencies, polarizations and
covariance terms it holds, choose a frequency for the session, and verify that pixels can be read.

NISAR's L-band radar can transmit two sub-bands at once: **frequency A** (here 20 MHz, fine
resolution) and **frequency B** (5 MHz, coarser, mainly for ionospheric correction). The file name
code `2005` means 20 + 5 MHz; `DHDH` means dual-pol H transmit, H and V receive, on both.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nisar_s1.config import load_kit
from nisar_s1 import product as P

kit = load_kit()                                  # notebooks/kit_config.yaml
OUT = Path(kit["data_dir"]) / "outputs" / "02"   # outputs stay outside the repository
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 90, "axes.spines.top": False, "axes.spines.right": False})
session = P.load_session(kit["session_file"], required=("granule", "gcov_local", "gcov_url"))
src_ctx = lambda: P.open_local_or_remote(session["gcov_local"], session["gcov_url"])

In [2]:
with src_ctx() as h5:
    prof = P.product_profile(h5)
print("polarimetric mode (from content):", prof.polarimetric_mode)
rows = []
for f, gr in prof.grids.items():
    rows.append({"frequency": f, "polarizations": gr.polarizations, "covariance terms": gr.terms,
                 "rows x cols": gr.shape, "pixel (m)": (gr.dx, -gr.dy), "EPSG": gr.epsg,
                 "complex terms stored": prof.complex_terms_available(f)})
pd.DataFrame(rows)

opening local file: NISAR_L2_PR_GCOV_034_144_A_026_2005_DHDH_M_20261103T041937_20261103T042013_P05036_N_F_J_001.h5
polarimetric mode (from content): dual-pol linear (HH/HV)


,frequency,polarizations,covariance terms,rows x cols,pixel (m),EPSG,complex terms stored
0,frequencyA,"[HH, HV]","[HHHH, HVHV]","(17928, 18108)","(20.0, 20.0)",32632,False
1,frequencyB,"[HH, HV]","[HHHH, HVHV]","(4482, 4527)","(80.0, 80.0)",32632,False


The name is only a **hint**: we cross-check it against the content. A mismatch (renamed file,
wrong download) is reported instead of silently trusted.

In [3]:
issues = P.check_name_consistency(prof, session["granule"])
print("name vs content:", "consistent" if not issues else issues)

name vs content: consistent


## Choose the session frequency
Rule used here: the frequency with the finest pixel (A). Change `FREQ` to work at 5 MHz instead
(notebook 09 compares the two).

In [4]:
FREQ = min(prof.grids, key=lambda f: abs(prof.grids[f].dx))
grid = prof.grids[FREQ]
session = P.save_session(kit["session_file"], frequency=FREQ, epsg=grid.epsg,
                         polarimetric_mode=prof.polarimetric_mode, terms=grid.terms)
print("session frequency:", FREQ, "| pixel", grid.dx, "m | EPSG", grid.epsg)

session frequency: frequencyA | pixel 20.0 m | EPSG 32632


## Verify pixel access (a 4 x 4 window at the grid centre)

In [5]:
cy, cx = grid.shape[0] // 2, grid.shape[1] // 2
centre = P.Window(slice(cy, cy + 4), slice(cx, cx + 4), grid.transform, f"EPSG:{grid.epsg}")
with src_ctx() as h5:
    for t in grid.terms:
        v = P.read_layer(h5, grid, t, centre)
        print(f"{t}: {np.nanmean(v):.4f} (linear power, mean of 16 pixels)")

opening local file: NISAR_L2_PR_GCOV_034_144_A_026_2005_DHDH_M_20261103T041937_20261103T042013_P05036_N_F_J_001.h5
HHHH: 0.0509 (linear power, mean of 16 pixels)
HVHV: 0.0083 (linear power, mean of 16 pixels)


### Checkpoint
How many frequencies? Which one did the session keep and why? Are the off-diagonal (complex)
covariance terms stored in this dual-pol GCOV? (Answer: no — notebook 13 explains how to get them.)

In [6]:
checks = {
    "two frequencies discovered": len(prof.grids) == 2,
    "HH and HV present": set(grid.polarizations) == {"HH", "HV"},
    "name consistent with content": not issues,
    "frequency saved": "frequency" in P.load_session(kit["session_file"]),
}
for k, v in checks.items():
    print(f"  [{'x' if v else ' '}] {k}")
assert all(checks.values()), "some checks failed: read the messages above"
print("Module 02 STATUS: PASS")

  [x] two frequencies discovered
  [x] HH and HV present
  [x] name consistent with content
  [x] frequency saved
Module 02 STATUS: PASS
